# fxbt2 — Full Library Tour

This notebook walks through every module in `fxbt2` using **live Bloomberg data** via `pdblp`.

Run this on a Windows machine with Bloomberg Terminal open.

| Module | What it does |
|---|---|
| `data.PdblpLoader` | Fetch OHLCV + forwards + vol + implied yield from Bloomberg |
| `data.ForwardBuilder` | Construct outrights and implied yield differentials |
| `signals.price` | Momentum, MA crossover, MACD, breakout, mean reversion, RSI |
| `signals.macro` | Carry (fwd pts), carry (implied yield), yield range, yield trend |
| `backtest.Backtest` | Run a full backtest — returns mode or notional USD mode |
| `backtest.positions` | Position sizers: fixed, vol-target, inv-vol, Kelly, equal-weight, VaR-target |
| `costs.model` | FixedSpreadModel (pip spreads) or SpreadCostModel (live bid/ask) |
| `metrics.stats` | Sharpe, Sortino, Calmar, MDD, hit rate, profit factor, VaR, CVaR |
| `portfolio.construction` | Net currency exposure, USD-cross aggregation |
| `portfolio.risk` | Rolling VaR, risk attribution per pair |
| `report.tearsheet` | 6-panel tear sheet: equity, drawdown, rolling Sharpe, monthly heatmap |
| `execution.trade_table` | Build today's trade delta table with IMM settlement date |
| `curves.CurveBuilder` | Fetch and plot the full forward curve (ON → 2Y) per pair |
| `curves.plot_*` | Curve snapshot, historical overlay, percentile bands, implied yields |

---


## 1 — Setup

Update `FXBT2_PATH` to the folder containing the `fxbt2/` directory.
Bloomberg Terminal must be running before executing the data cells.


In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

# ── Path setup ────────────────────────────────────────────────────────────────
FXBT2_PATH = r"C:\Users\your_username\fxbt2_parent"   # <-- update this
sys.path.insert(0, FXBT2_PATH)

# ── Verify imports ────────────────────────────────────────────────────────────
from fxbt2.data.pdblp_loader import PdblpLoader
from fxbt2.data.forward_builder import ForwardBuilder
from fxbt2.signals import price as price_signals
from fxbt2.signals import macro as macro_signals
from fxbt2.backtest.engine import Backtest
from fxbt2.backtest.positions import (
    fixed_size, vol_target, inverse_vol_weight, kelly, equal_weight, var_target,
)
from fxbt2.costs.model import FixedSpreadModel, SpreadCostModel
from fxbt2.metrics.stats import summary, compare, sharpe, max_drawdown
from fxbt2.portfolio.construction import generate_pairs, net_ccy_exposure, net_ccy_exposure_usd
from fxbt2.portfolio.risk import rolling_var, risk_attribution
from fxbt2.report.tearsheet import plot_tearsheet
from fxbt2.execution.trade_table import build_trade_table
from fxbt2.curves import (
    CurveBuilder, TENORS, build_tenor_tickers,
    plot_curve_today, plot_curve_history, plot_percentile_bands,
    plot_implied_yields, plot_curve_heatmap, plot_curve_dashboard,
)

print("All imports OK")


---
## 2 — Connect to Bloomberg

`PdblpLoader` wraps `pdblp.BCon` and connects automatically on first use.
It fetches OHLCV, 1M forward points, 1M ATM implied vol, and 1M implied yield
in a single call.


In [ ]:
# ── Configure ────────────────────────────────────────────────────────────────
PAIRS  = ["EURUSD", "USDJPY", "GBPUSD", "USDMXN", "USDKRW", "USDINR"]
START  = "20230101"
END    = "20250101"
FIXING = "CMPT"

loader = PdblpLoader(fixing=FIXING, load_forwards=True, load_vol=True, load_yield=True)

print(f"Loading {len(PAIRS)} pairs from {START} to {END} ...")
raw = loader.load(PAIRS, start=START, end=END)

print(f"\nShape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
print(f"Date range: {raw.index.min()} → {raw.index.max()}")
raw.tail(3)


In [ ]:
# ── Pivot to wide format for signal generation ────────────────────────────────
prices     = raw.pivot(columns="pair", values="close")
fwd_points = raw.pivot(columns="pair", values="fwd_points")
impl_yield = raw.pivot(columns="pair", values="impl_yield")
impl_vol   = raw.pivot(columns="pair", values="impl_vol")

print("Wide prices shape:", prices.shape)
print("Pairs:", list(prices.columns))
prices.tail(3).round(4)


---
## 3 — Forward Builder

`ForwardBuilder` constructs 1M forward outrights and annualised implied yield
differentials from raw spot + forward point data.

Handles all quoting conventions automatically:
- EUR/GBP/AUD/NZD: USD-per-CCY, fwd pts inverted
- JPY/CAD/CHF/EM: USD-per-CCY conventional, divide by 100 or 10000
- NDF pairs (KRW, INR, IDR, TWD, PHP, BRL, etc.): outrights fetched directly


In [ ]:
fb = ForwardBuilder(ann_factor=12)

# For pairs where we have spot + fwd_pts (deliverable pairs)
DEL_PAIRS = ["EURUSD", "USDJPY", "GBPUSD", "USDMXN"]
CCY_CODES  = list({p[:3] for p in DEL_PAIRS} | {p[3:] for p in DEL_PAIRS})

# Extract ccy-level spot (EURUSD close = EUR spot in USD terms)
spot_ccy = pd.DataFrame({
    "EUR": prices["EURUSD"],
    "GBP": prices["GBPUSD"],
    "JPY": 1 / prices["USDJPY"],   # convert to USD per JPY convention
    "MXN": 1 / prices["USDMXN"],
    "USD": pd.Series(1.0, index=prices.index),
})
fwd_ccy = pd.DataFrame({
    "EUR": fwd_points["EURUSD"],
    "GBP": fwd_points["GBPUSD"],
    "JPY": fwd_points["USDJPY"],
    "MXN": fwd_points["USDMXN"],
    "USD": pd.Series(0.0, index=prices.index),
})

# Implied yield differential per pair
impl_yield_built = fb.build_implied_yield(spot_ccy, fwd_ccy, list(spot_ccy.columns), DEL_PAIRS)

print("Implied yield differential (ann.) — last 5 rows:")
(impl_yield_built * 100).round(3).tail()


---
## 4 — Price Signals

All signal functions share the same interface:
- **Input**: wide prices DataFrame (index = DatetimeIndex, columns = pairs)
- **Output**: same-shape DataFrame, values in {+1, 0, -1} or continuous
- **Forward-safe**: `.shift(1)` applied internally — no look-ahead

| Signal | Logic |
|---|---|
| `momentum` | Sign / z-score / cross-sectional rank of N-period return |
| `crossover` | +1 when fast MA > slow MA |
| `macd_signal` | MACD histogram crossover |
| `breakout` | Donchian channel: +1 above N-day high, -1 below N-day low |
| `mean_reversion` | Z-score entry/exit: buy below -z_entry, sell above +z_entry |
| `rsi_signal` | RSI: buy oversold, sell overbought |
| `vol_regime` | Filter: 1 = normal vol (allow trades), 0 = high vol (suppress) |


In [ ]:
# ── Momentum (20-day, sign) ───────────────────────────────────────────────────
sig_mom = price_signals.momentum(prices, lookback=20, signal_type="sign")

# ── MA crossover (10/50) ──────────────────────────────────────────────────────
sig_cross = price_signals.crossover(prices, fast=10, slow=50)

# ── MACD ─────────────────────────────────────────────────────────────────────
sig_macd = price_signals.macd_signal(prices, fast=12, slow=26, signal_period=9)

# ── Breakout (20-day Donchian) ────────────────────────────────────────────────
sig_breakout = price_signals.breakout(prices, lookback=20)

# ── Mean reversion (z-score) ──────────────────────────────────────────────────
sig_mr = price_signals.mean_reversion(prices, lookback=20, z_entry=1.5, z_exit=0.5)

# ── RSI ───────────────────────────────────────────────────────────────────────
sig_rsi = price_signals.rsi_signal(prices, period=14, overbought=70, oversold=30)

# ── Vol regime filter ─────────────────────────────────────────────────────────
vol_filter = price_signals.vol_regime(prices, lookback=20, high_vol_percentile=75)

print("Momentum signal — last 5 rows:")
print(sig_mom.tail())

print("\nActive signals today (momentum):")
print(sig_mom.iloc[-1])


In [ ]:
# ── Visualise signal hit rate vs no-signal periods ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
axes = axes.flatten()

signals_to_plot = [
    (sig_mom,      "Momentum (20d)"),
    (sig_cross,    "MA Crossover (10/50)"),
    (sig_macd,     "MACD"),
    (sig_breakout, "Breakout (20d)"),
    (sig_mr,       "Mean Reversion"),
    (sig_rsi,      "RSI"),
]

pair_to_show = "EURUSD"
for ax, (sig, name) in zip(axes, signals_to_plot):
    s = sig[pair_to_show].dropna()
    ax.plot(s.index, s.values, lw=0.8, color="steelblue")
    ax.axhline(0, color="black", lw=0.5, linestyle=":")
    ax.axhline(1, color="green", lw=0.5, linestyle="--", alpha=0.5)
    ax.axhline(-1, color="crimson", lw=0.5, linestyle="--", alpha=0.5)
    ax.set_title(f"{pair_to_show} — {name}", fontweight="bold")
    ax.grid(True, alpha=0.2)
    ax.set_yticks([-1, 0, 1])

plt.suptitle("Price Signal Comparison — EURUSD", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 5 — Macro / Carry Signals

Signals derived from interest rate differentials embedded in FX forwards.

| Signal | Logic |
|---|---|
| `carry_fwd` | Annualised carry from 1M forward points. +1 = positive carry (sell base) |
| `carry_yield` | Carry from implied yield differential directly (more accurate for EM/NDF) |
| `yield_range` | Enter when yield differential is at historical extremes (percentile rank) |
| `yield_trend` | Continuous z-score of EMA-smoothed yield differential — ride rate trends |


In [ ]:
# ── Carry from forward points ─────────────────────────────────────────────────
sig_carry_fwd = macro_signals.carry_fwd(fwd_points, prices, signal_type="sign")

# ── Carry from implied yield (use loader's impl_yield if available) ────────────
sig_carry_yield = macro_signals.carry_yield(impl_yield, signal_type="sign")

# ── Yield range: enter when carry is at historical high/low ──────────────────
sig_yield_range = macro_signals.yield_range(impl_yield, lookback=252)

# ── Yield trend: continuous signal riding the rate differential trend ─────────
sig_yield_trend = macro_signals.yield_trend(impl_yield, lookback=252, ema_span=5)

print("Carry signal (from fwd pts) — last 5 rows:")
print(sig_carry_fwd.tail())

print("\nYield trend signal (continuous) — last 5 rows:")
print(sig_yield_trend.tail().round(3))


In [ ]:
# ── Compare carry vs momentum on the same pair ───────────────────────────────
pair = "USDMXN"
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(prices[pair].dropna(), lw=1, color="black")
axes[0].set_title(f"{pair} Spot", fontweight="bold")
axes[0].set_ylabel("Spot Rate")
axes[0].grid(True, alpha=0.25)

axes[1].plot(sig_mom[pair].dropna(), lw=1, color="steelblue", label="Momentum")
axes[1].axhline(0, color="black", lw=0.5, linestyle=":")
axes[1].set_title("Momentum Signal", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.25)

axes[2].plot(sig_carry_yield[pair].dropna(), lw=1, color="darkorange", label="Carry (yield)")
axes[2].plot(sig_yield_trend[pair].dropna(), lw=1, color="purple", alpha=0.7, label="Yield trend")
axes[2].axhline(0, color="black", lw=0.5, linestyle=":")
axes[2].set_title("Carry Signals", fontweight="bold")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


---
## 6 — Backtest Engine

`Backtest` takes a prices DataFrame and a signals DataFrame and runs a vectorised backtest.

**Two modes:**
- `pnl_mode='returns'` — positions are portfolio weights, PnL in % per period
- `pnl_mode='notional'` — positions are USD notional, PnL in absolute USD

Cost model (`FixedSpreadModel`) applies bid/ask spread + optional slippage on every trade,
and rollover financing cost (from implied yield or forward points) each period.


In [ ]:
# ── Returns mode: simple momentum strategy ────────────────────────────────────
bt = Backtest(
    data=prices,
    signals=sig_mom,
    cost_model=FixedSpreadModel(),          # default pip spreads per pair
    sizer=lambda s, p: fixed_size(s, size=1.0),
    freq="1D",
    name="Momentum (20d)",
    pnl_mode="returns",
)

result = bt.run()
print("Backtest complete")
print(result.summary())


In [ ]:
# ── Notional mode: carry strategy, $10M per signal ───────────────────────────
bt_notional = Backtest(
    data=prices,
    signals=sig_carry_yield,
    cost_model=FixedSpreadModel(),
    sizer=lambda s, p: fixed_size(s, size=1.0),
    freq="1D",
    name="Carry (yield, $10M)",
    pnl_mode="notional",
    notional_size=10_000_000,
    impl_yield=impl_yield,
)

result_notional = bt_notional.run()
print(result_notional.summary())


In [ ]:
# ── Quick equity curve comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

r1 = result
r2 = result_notional

# Returns mode
axes[0].plot(r1.equity_curve.dropna(), lw=1.5, color="steelblue")
axes[0].axhline(1, color="black", lw=0.7, linestyle=":")
axes[0].set_title(f"{r1.metadata['name']} — Equity", fontweight="bold")
axes[0].set_ylabel("Equity (1 = start)")
axes[0].grid(True, alpha=0.25)

# Notional mode
axes[1].plot(r2.equity_curve.dropna(), lw=1.5, color="darkorange")
axes[1].axhline(0, color="black", lw=0.7, linestyle=":")
axes[1].set_title(f"{r2.metadata['name']} — Cumulative PnL", fontweight="bold")
axes[1].set_ylabel("Cumulative PnL (USD)")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


---
## 7 — Position Sizers

Five sizing methods — each takes a signal DataFrame and returns a sized position DataFrame.

| Sizer | Logic |
|---|---|
| `fixed_size` | +/- constant size per signal |
| `vol_target` | Scale each pair so realised vol contribution = target / n_pairs |
| `inverse_vol_weight` | Weight proportional to 1/vol — rebalances weekly by default |
| `kelly` | Fractional Kelly: mean_return / variance × fraction |
| `equal_weight` | 1/n_active each period |
| `var_target` | Scale entire portfolio so rolling VaR = USD target (notional mode) |


In [ ]:
# ── Compare all sizers on momentum signal ────────────────────────────────────
sizers = {
    "Fixed (1x)":      lambda s, p: fixed_size(s, size=1.0),
    "Vol-Target 10%":  lambda s, p: vol_target(s, p, target_vol=0.10, freq="1D"),
    "Inv-Vol (Mon)":   lambda s, p: inverse_vol_weight(s, p, lookback=22, rebalance_day=0),
    "Kelly 0.5x":      lambda s, p: kelly(s, p, lookback=60, fraction=0.5),
    "Equal Weight":    lambda s, p: equal_weight(s),
}

results_by_sizer = {}
for name, sizer_fn in sizers.items():
    bt_s = Backtest(
        data=prices,
        signals=sig_mom,
        cost_model=FixedSpreadModel(),
        sizer=sizer_fn,
        freq="1D",
        name=name,
        pnl_mode="returns",
    )
    results_by_sizer[name] = bt_s.run()

# Side-by-side summary
frames = [r.summary() for r in results_by_sizer.values()]
comparison = pd.concat(frames, axis=1)
comparison.columns = list(sizers.keys())
comparison


In [ ]:
# ── Equity curves for all sizers ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10.colors

for i, (name, r) in enumerate(results_by_sizer.items()):
    eq = r.equity_curve.dropna()
    ax.plot(eq.index, eq.values, lw=1.5, label=name, color=colors[i])

ax.axhline(1, color="black", lw=0.7, linestyle=":")
ax.set_title("Momentum Signal — Equity Curve by Sizer", fontweight="bold")
ax.set_ylabel("Equity (1 = start)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


---
## 8 — Signal Combination

Combine multiple signals by averaging — the simplest ensemble approach.
Use the vol regime filter to suppress signals during high-volatility periods.


In [ ]:
# ── Average momentum + carry + vol filter ────────────────────────────────────
sig_combined = (sig_mom + sig_carry_yield) / 2

# Apply vol regime filter: zero out signals when vol is in the top quartile
sig_filtered = sig_combined * vol_filter

bt_combined = Backtest(
    data=prices,
    signals=sig_combined,
    cost_model=FixedSpreadModel(),
    sizer=lambda s, p: vol_target(s, p, target_vol=0.10, freq="1D"),
    freq="1D",
    name="Combined (Mom + Carry)",
    pnl_mode="returns",
)

bt_filtered = Backtest(
    data=prices,
    signals=sig_filtered,
    cost_model=FixedSpreadModel(),
    sizer=lambda s, p: vol_target(s, p, target_vol=0.10, freq="1D"),
    freq="1D",
    name="Combined + Vol Filter",
    pnl_mode="returns",
)

r_combined = bt_combined.run()
r_filtered = bt_filtered.run()

# Compare
compare(
    (result.returns,     "Momentum only"),
    (r_combined.returns, "Mom + Carry"),
    (r_filtered.returns, "Mom + Carry + Vol Filter"),
    freq="1D",
)


---
## 9 — Full Tear Sheet

6-panel tear sheet: equity curve, drawdown, rolling Sharpe (52-week),
monthly returns heatmap, per-pair contribution, and summary stats table.


In [ ]:
# Full tear sheet for the combined strategy
r_combined.tearsheet()


In [ ]:
# You can also call individual panels
from fxbt2.report.tearsheet import (
    plot_equity_curve, plot_drawdown, plot_rolling_sharpe, plot_monthly_heatmap,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
plot_equity_curve(r_combined, ax=axes[0, 0])
plot_drawdown(r_combined, ax=axes[0, 1])
plot_rolling_sharpe(r_combined, window=52, ax=axes[1, 0])
plot_monthly_heatmap(r_combined, ax=axes[1, 1])
plt.tight_layout()
plt.show()


---
## 10 — Metrics

All metrics are importable individually.

| Function | Returns |
|---|---|
| `sharpe` | Annualised Sharpe ratio |
| `sortino` | Sortino ratio (downside vol only) |
| `calmar` | Calmar ratio (ann. return / max drawdown) |
| `max_drawdown` | Maximum peak-to-trough drawdown |
| `hit_rate` | % of periods with positive return |
| `profit_factor` | Gross wins / gross losses |
| `var` / `cvar` | Value at Risk at given confidence level |
| `summary` | Full table of all stats |
| `compare` | Side-by-side multi-strategy comparison |


In [ ]:
from fxbt2.metrics.stats import (
    annualised_return, annualised_vol, sharpe, sortino, calmar,
    max_drawdown, hit_rate, profit_factor, var, cvar, drawdown_series,
)

r = r_combined.returns.dropna()

print(f"Ann. Return:    {annualised_return(r):.2%}")
print(f"Ann. Vol:       {annualised_vol(r):.2%}")
print(f"Sharpe:         {sharpe(r):.2f}")
print(f"Sortino:        {sortino(r):.2f}")
print(f"Calmar:         {calmar(r):.2f}")
print(f"Max Drawdown:   {max_drawdown(r):.2%}")
print(f"Hit Rate:       {hit_rate(r):.2%}")
print(f"Profit Factor:  {profit_factor(r):.2f}")
print(f"VaR 95%:        {var(r, 0.05):.2%}")
print(f"CVaR 95%:       {cvar(r, 0.05):.2%}")


In [ ]:
# Rolling Sharpe time series
from fxbt2.metrics.stats import _ann
import numpy as np

window = 52
af = _ann("1D")
rs = r_combined.rolling_sharpe(window=window).dropna()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rs.index, rs.values, lw=1.5, color="darkorange")
ax.axhline(0, color="black", lw=0.8, linestyle=":")
ax.axhline(1, color="green", lw=0.8, linestyle="--", alpha=0.6, label="Sharpe = 1")
ax.fill_between(rs.index, rs.values, 0,
                where=rs > 0, color="steelblue", alpha=0.2, label="Positive")
ax.fill_between(rs.index, rs.values, 0,
                where=rs < 0, color="crimson", alpha=0.2, label="Negative")
ax.set_title(f"Rolling {window}-period Sharpe — {r_combined.metadata['name']}", fontweight="bold")
ax.set_ylabel("Sharpe Ratio")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


---
## 11 — Portfolio Construction

`generate_pairs` generates all unique FX pairs from a list of currencies.

`net_ccy_exposure` and `net_ccy_exposure_usd` aggregate pair-level positions
into single-currency net exposures — essential for risk monitoring and execution.


In [ ]:
# ── Generate all pairs from a currency basket ────────────────────────────────
basket = ["USD", "EUR", "GBP", "JPY", "MXN"]
all_pairs = generate_pairs(basket)
print(f"All pairs from {basket}:")
print(all_pairs)


In [ ]:
# ── Net currency exposure from today's positions ─────────────────────────────
# Use today's position sizes (USD notional from notional backtest)
today_pos = result_notional.positions.iloc[[-1]]

net_ccy = net_ccy_exposure(today_pos)
print("Net currency exposure (USD notional):")
print(net_ccy.T.rename(columns={net_ccy.index[0]: "Net USD"}).round(0))

net_usd_cross = net_ccy_exposure_usd(today_pos)
print("\nNetted as USD crosses:")
print(net_usd_cross.T.rename(columns={net_usd_cross.index[0]: "To Hedge"}).round(0))


---
## 12 — Portfolio Risk

`risk_attribution` decomposes portfolio variance into per-pair contributions.
`rolling_var` computes rolling historical VaR (parametric normal).


In [ ]:
# ── Risk attribution ─────────────────────────────────────────────────────────
attr = risk_attribution(r_combined.pair_returns, freq="1D")
print("Risk attribution:")
print(attr.sort_values("pct_vol_contrib", ascending=False))


In [ ]:
# ── Plot: vol contribution vs Sharpe ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vol contribution bar
attr_sorted = attr.sort_values("pct_vol_contrib", ascending=True)
colors = ["crimson" if v < 0 else "steelblue" for v in attr_sorted["sharpe"]]
axes[0].barh(attr_sorted.index, attr_sorted["pct_vol_contrib"] * 100, color=colors)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("% Vol Contribution per Pair", fontweight="bold")
axes[0].set_xlabel("% of Portfolio Variance")
axes[0].grid(True, alpha=0.25, axis="x")

# Sharpe bar
attr_sharpe = attr.sort_values("sharpe", ascending=True)
colors2 = ["crimson" if v < 0 else "steelblue" for v in attr_sharpe["sharpe"]]
axes[1].barh(attr_sharpe.index, attr_sharpe["sharpe"], color=colors2)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Sharpe Ratio per Pair", fontweight="bold")
axes[1].set_xlabel("Sharpe Ratio")
axes[1].grid(True, alpha=0.25, axis="x")

plt.suptitle("Risk Attribution", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Rolling 95% VaR ──────────────────────────────────────────────────────────
# Convert returns to notional-like PnL for VaR display
pnl_scaled = r_combined.returns * 10_000_000   # hypothetical $10M portfolio

rv = rolling_var(pnl_scaled.cumsum(), window=260, confidence=0.05)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(pnl_scaled.index, pnl_scaled.values, 0,
                where=pnl_scaled >= 0, color="steelblue", alpha=0.3, label="Daily PnL")
ax.fill_between(pnl_scaled.index, pnl_scaled.values, 0,
                where=pnl_scaled < 0, color="crimson", alpha=0.3)
ax.plot(rv.index, rv.values, color="black", lw=1.5, linestyle="--", label="Rolling 95% VaR")
ax.axhline(0, color="black", lw=0.7)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Daily PnL vs Rolling 95% VaR ($10M portfolio)", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


---
## 13 — Walk-Forward Analysis

`Backtest.walk_forward()` runs a rolling OOS validation.
The `signal_fn` receives the training window and test window — you fit parameters
on training data and generate signals on the test data.


In [ ]:
# ── Walk-forward: refit momentum lookback each year ─────────────────────────
def signal_fn_momentum(train_prices, test_prices):
    """Use 20-day momentum signal on test data (could refit lookback on train)."""
    return price_signals.momentum(test_prices, lookback=20, signal_type="sign")

bt_wf = Backtest(
    data=prices,
    signals=sig_mom,           # initial signals (overridden by walk_forward)
    cost_model=FixedSpreadModel(),
    sizer=lambda s, p: vol_target(s, p, target_vol=0.10, freq="1D"),
    freq="1D",
    name="Momentum (Walk-Forward)",
    pnl_mode="returns",
)

result_wf = bt_wf.walk_forward(
    train_periods=252,      # 1 year in-sample
    test_periods=63,        # 3 months out-of-sample
    signal_fn=signal_fn_momentum,
)

print("Walk-forward complete")
print(result_wf.summary())


In [ ]:
# Compare IS vs OOS performance
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(result.equity_curve.dropna(), lw=1.5, color="steelblue", alpha=0.5, label="In-sample (full)")
ax.plot(result_wf.equity_curve.dropna(), lw=1.5, color="darkorange", label="Walk-forward OOS")
ax.axhline(1, color="black", lw=0.7, linestyle=":")
ax.set_title("Momentum — In-Sample vs Walk-Forward OOS", fontweight="bold")
ax.set_ylabel("Equity (1 = start)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


---
## 14 — Execution Trade Table

`build_trade_table` computes today's delta-to-trade vs yesterday's positions.
Pass a live `pdblp.BCon` connection to fetch current rates and IMM settlement date.


In [ ]:
# ── Build trade table from today's positions ────────────────────────────────
# Use last 2 rows of notional positions
pos_last2 = result_notional.positions.iloc[-2:]

# Without live connection: ref_rates comes from last known prices
ref_rates = prices.iloc[-2:]

trade_table = build_trade_table(
    tradesize=pos_last2,
    ref_rates=ref_rates,
    ndf_pairs=["USDKRW", "USDINR"],
)

print("Trade table (what to execute today):")
print(trade_table.round(0))


In [ ]:
# ── With live Bloomberg connection (uncomment when Terminal is running) ──────
# import pdblp
# con = pdblp.BCon(debug=False, port=8194, timeout=50000)
# con.start()
#
# trade_table_live = build_trade_table(
#     tradesize=pos_last2,
#     pdblp_connection=con,
#     ndf_pairs=["USDKRW", "USDINR"],
# )
# print(trade_table_live.round(0))


---
## 15 — Forward Curve Analysis

`CurveBuilder` fetches the full tenor structure (ON → 2Y) for any pair.
Works with `pdblp.BCon` (Terminal) locally, or `bql.Service()` in BQuant.

Seven plot functions cover all curve analysis needs.


In [ ]:
# ── Fetch today's curve for USDKRW ─────────────────────────────────────────
import pdblp
con = pdblp.BCon(debug=False, port=8194, timeout=50000)
con.start()

cb = CurveBuilder(fixing="CMPT")

CURVE_PAIR = "USDKRW"
curve_today = cb.fetch_curve(CURVE_PAIR, con)

print(f"{CURVE_PAIR} — Forward Curve today:")
print(curve_today.round(2).to_string())


In [ ]:
# ── Plot today's curve ───────────────────────────────────────────────────────
plot_curve_today(curve_today, pair=CURVE_PAIR)


In [ ]:
# ── Fetch 2-year curve history ───────────────────────────────────────────────
print(f"Fetching {CURVE_PAIR} curve history {START} → {END} ...")
curve_history = cb.fetch_history(CURVE_PAIR, con, start=START, end=END)

print(f"Shape: {curve_history.shape}")
print(f"Columns: {list(curve_history.columns)}")
curve_history.tail(3).round(2)


In [ ]:
# ── Historical overlay: today vs 1M/3M/6M/1Y ago ────────────────────────────
plot_curve_history(curve_history, pair=CURVE_PAIR)


In [ ]:
# ── Percentile fan chart: cheap or rich vs history? ─────────────────────────
plot_percentile_bands(curve_history, pair=CURVE_PAIR)


In [ ]:
# ── Implied yield term structure ────────────────────────────────────────────
plot_implied_yields(curve_history, pair=CURVE_PAIR)


In [ ]:
# ── Level heatmap over time ──────────────────────────────────────────────────
plot_curve_heatmap(curve_history, pair=CURVE_PAIR)


In [ ]:
# ── 4-panel dashboard (all views at once) ────────────────────────────────────
plot_curve_dashboard(curve_history, pair=CURVE_PAIR)


In [ ]:
# ── Implied yield curve (tabular) ────────────────────────────────────────────
yield_curve = cb.implied_yield_curve(curve_history)
print("Implied yields (annualised %) — last 5 rows:")
(yield_curve * 100).round(3).tail()


In [ ]:
# ── Check Bloomberg tickers for any pair ────────────────────────────────────
for pair in ["USDKRW", "USDINR", "EURUSD", "USDJPY"]:
    tickers = build_tenor_tickers(pair, fixing="CMPT")
    print(f"\n{pair}:")
    for tenor, ticker in tickers.items():
        print(f"  {tenor:4s}  {ticker}")


In [ ]:
# Close Bloomberg connection when done
con.stop()
print("Bloomberg connection closed")


---
## 16 — Cost Models

Two cost models are provided. Both handle bid/ask spread and optional rollover financing.

**`FixedSpreadModel`** — default, uses pre-calibrated pip spreads per pair.
Override any pair: `FixedSpreadModel(spreads_pips={"EURUSD": 0.3})`

**`SpreadCostModel`** — uses actual bid/ask data when available (from PdblpLoader).


In [ ]:
# ── Inspect default spreads ──────────────────────────────────────────────────
from fxbt2.costs.model import FixedSpreadModel
cm = FixedSpreadModel()

print("Default pip spreads:")
for pair, pips in sorted(cm.spreads.items()):
    print(f"  {pair:12s}  {pips:.1f} pips")


In [ ]:
# ── Compare cost impact: tight vs wide spreads ───────────────────────────────
bt_tight = Backtest(
    data=prices, signals=sig_mom,
    cost_model=FixedSpreadModel(default_spread_pips=0.5),
    sizer=lambda s, p: fixed_size(s, 1.0),
    freq="1D", name="Tight spreads (0.5 pip)", pnl_mode="returns",
)

bt_wide = Backtest(
    data=prices, signals=sig_mom,
    cost_model=FixedSpreadModel(default_spread_pips=3.0),
    sizer=lambda s, p: fixed_size(s, 1.0),
    freq="1D", name="Wide spreads (3 pip)", pnl_mode="returns",
)

rt = bt_tight.run()
rw = bt_wide.run()

compare(
    (rt.returns, "Tight spreads"),
    (rw.returns, "Wide spreads"),
    freq="1D",
)


---
## Summary — What's in fxbt2

```
fxbt2/
├── data/
│   ├── PdblpLoader      — Bloomberg Terminal (pdblp), OHLCV + fwd pts + vol + yield
│   ├── BQuantLoader     — Bloomberg BQuant (bql), same output schema
│   ├── CSVLoader        — offline CSV files
│   ├── ForwardBuilder   — construct outrights + implied yields from raw data
│   └── ticker_dict      — Bloomberg ticker mappings for 30+ currencies
│
├── signals/
│   ├── price            — momentum, crossover, MACD, breakout, mean_reversion, RSI, vol_regime
│   └── macro            — carry_fwd, carry_yield, yield_range, yield_trend
│
├── backtest/
│   ├── Backtest         — vectorised engine, returns + notional mode, walk-forward
│   └── positions        — fixed_size, vol_target, inverse_vol_weight, kelly, equal_weight, var_target
│
├── costs/
│   └── FixedSpreadModel / SpreadCostModel — spread + slippage + rollover
│
├── metrics/
│   └── stats            — sharpe, sortino, calmar, MDD, hit_rate, profit_factor, VaR, CVaR
│
├── portfolio/
│   ├── construction     — generate_pairs, net_ccy_exposure, net_ccy_exposure_usd
│   └── risk             — rolling_var, risk_attribution
│
├── report/
│   └── tearsheet        — 6-panel tear sheet + individual plot functions
│
├── execution/
│   └── trade_table      — delta-to-trade table with IMM settlement date
│
└── curves/
    ├── CurveBuilder     — fetch full forward curve (ON → 2Y) via pdblp or bql
    └── plot_*           — curve_today, curve_history, percentile_bands,
                           implied_yields, heatmap, dashboard, tenor_vs_spot
```
